# D141 — ER Modeling, Keys, and Normalization

This notebook designs a small but realistic e-commerce sales database. It moves from business requirements to an entity-relationship model, implements key choices, and uses runnable SQL to explain normalization.

The goal is not to memorize rules. The goal is to recognize repeated facts, dependencies, and relationship patterns so each fact has one reliable home.

## Setup and connection

**InnoDB** is MySQL's default storage engine. It supports transactions, foreign keys, row-level locking, and crash recovery, making it suitable for transactional sales data.

In [ ]:
import os
import mysql.connector
from mysql.connector import Error

# Local learning values. Move credentials to environment variables later.
MYSQL_HOSTNAME = "localhost"
MYSQL_PORT = 3306
MYSQL_USERNAME = "root"
MYSQL_PASSWORD = "root"
MYSQL_DATABASE = "salesdb"

# Recommended later:
# MYSQL_HOSTNAME = os.environ["MYSQL_HOSTNAME"]
# MYSQL_PORT = int(os.environ.get("MYSQL_PORT", "3306"))
# MYSQL_USERNAME = os.environ["MYSQL_USERNAME"]
# MYSQL_PASSWORD = os.environ["MYSQL_PASSWORD"]
# MYSQL_DATABASE = os.environ.get("MYSQL_DATABASE", "salesdb")

### Re-create the learning database

Connect to the MySQL server without selecting a database. For a clean classroom run, this cell drops the learning database if it exists and creates it again. Run it only when the existing `salesdb` data is no longer needed.

In [ ]:
server_connection = mysql.connector.connect(
    host=MYSQL_HOSTNAME,
    port=MYSQL_PORT,
    user=MYSQL_USERNAME,
    password=MYSQL_PASSWORD,
)
server_cursor = server_connection.cursor()

server_cursor.execute(f"DROP DATABASE IF EXISTS `{MYSQL_DATABASE}`")

query = f"""
CREATE DATABASE `{MYSQL_DATABASE}`
CHARACTER SET utf8mb4
COLLATE utf8mb4_unicode_ci
"""
server_cursor.execute(query)
server_cursor.close()
server_connection.close()
print(f"Database {MYSQL_DATABASE!r} is ready.")

### Connect to `salesdb`

This connection remains open while the notebook runs. Every later cursor is created from it and closed by the helper.

In [ ]:
connection = mysql.connector.connect(
    host=MYSQL_HOSTNAME,
    port=MYSQL_PORT,
    user=MYSQL_USERNAME,
    password=MYSQL_PASSWORD,
    database=MYSQL_DATABASE,
)
print("Connected:", connection.is_connected())
print("MySQL version:", connection.get_server_info())

### Reusable SQL executor

`execute_sql` prints returned rows for queries and affected-row counts for data-definition or data-change statements. It commits successful changes, rolls back failures, and consumes every result set.

In [ ]:
def print_rows(columns, rows):
    if not rows:
        print("No rows returned.")
        return
    print(" | ".join(columns))
    print("-+-".join("-" * len(column) for column in columns))
    for row in rows:
        print(" | ".join(str(value) for value in row))


def execute_sql(sql_text, params=None, *, many=False):
    """Execute one SQL statement and print results or the affected-row count."""
    cursor = connection.cursor()
    try:
        if many:
            cursor.executemany(sql_text, params or [])
        else:
            cursor.execute(sql_text, params or ())

        result_sets = []
        affected_rows = cursor.rowcount
        while True:
            if cursor.with_rows:
                columns = [item[0] for item in cursor.description]
                rows = cursor.fetchall()
                print_rows(columns, rows)
                result_sets.append(rows)
            if not cursor.nextset():
                break

        if result_sets:
            return result_sets[0] if len(result_sets) == 1 else result_sets
        connection.commit()
        print(f"Rows impacted: {affected_rows}")
        return affected_rows
    except Error:
        connection.rollback()
        raise
    finally:
        cursor.close()

## From business story to ER model

An **entity-relationship model** describes the important business things, their attributes, and how they relate before implementation details take over.

For this e-commerce example:

- A customer can save multiple addresses and place multiple sales orders.
- A category groups many products; each product belongs to one category in this simplified model.
- A sale contains one or more products through `sale_items`.
- A sale can have multiple shipments because items may ship separately.
- Each shipment goes to one saved address.

An **entity** usually becomes a table, an **attribute** becomes a column, and an entity occurrence becomes a row. Relationships become foreign keys or associative tables.

### Conceptual, logical, and physical models

- The **conceptual model** names business entities and relationships: Customer places Sale; Sale contains Product.
- The **logical model** adds attributes, identifiers, cardinality, and optionality without focusing on a particular database engine.
- The **physical model** chooses MySQL types, indexes, constraint names, and InnoDB implementation details.

Keeping these levels distinct helps business experts validate meaning before engineers optimize storage.

### Cardinality and optionality

Cardinality asks **how many**: one-to-one, one-to-many, or many-to-many. Optionality asks whether participation is required.

```text
customers 1 ─────< addresses
customers 1 ─────< sales
categories 1 ─────< products
sales      1 ─────< sale_items >───── 1 products
sales      1 ─────< shipments  >───── 1 addresses
```

`sale_items` resolves the many-to-many relationship between sales and products. A sale must have a customer, while a customer may exist before placing any sale.

### Attributes and relationship rules

Choose attributes that describe one entity and enforce important rules close to the data. Examples include a unique customer email, a positive product price, and a shipment status restricted to known values.

Some rules span multiple rows or involve timing and are better enforced by transactions or application services. An ER model communicates the rule even when one constraint cannot express all of it.

## Keys

A **key** identifies rows or connects related rows. Good key choices stabilize joins, prevent duplicates, and express business rules.

- A **superkey** is any column set that uniquely identifies a row, even if it contains unnecessary columns.
- A **candidate key** is a minimal superkey. A table can have several candidates.
- The chosen candidate is the **primary key**; remaining candidates are **alternate keys**, normally enforced with `UNIQUE`.
- A **composite key** contains multiple columns.
- A **foreign key** references a candidate key—normally another table's primary key—and protects referential integrity.

### Business, natural, surrogate, and sequence keys

A **natural key** comes from the domain, such as an officially assigned tax identifier. A **business key** is a meaningful identifier used by the organization, such as `customer_number` or SKU. It may be natural or organization-assigned.

A **surrogate key** is generated only to identify a database row. Common forms include:

- Integer sequence-style keys: compact and efficient. MySQL commonly uses `AUTO_INCREMENT`; unlike some systems, MySQL does not provide general standalone sequence objects.
- Random UUIDs: easy to generate across systems but larger and less index-friendly.
- Time-ordered UUIDs such as UUIDv7: distributed generation with better index locality.
- Snowflake-style 64-bit IDs: combine time and generator information; generation normally belongs to an application or service.

Surrogate keys do not replace business uniqueness. This model uses an integer surrogate primary key plus `UNIQUE` business keys.

### Create the `customers` entity

`customer_id` is a surrogate sequence-style key. `customer_number` and `email` are business candidate keys enforced as alternate keys.

In [ ]:
query = f"""
CREATE TABLE customers (
    customer_id BIGINT AUTO_INCREMENT PRIMARY KEY,
    customer_number VARCHAR(20) NOT NULL UNIQUE,
    customer_name VARCHAR(120) NOT NULL,
    email VARCHAR(255) NOT NULL UNIQUE,
    created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
) ENGINE=InnoDB
"""
execute_sql(query)

### Create the `addresses` entity

One customer can own many addresses. The foreign key expresses that relationship, while the address surrogate key gives shipments a stable reference.

In [ ]:
query = f"""
CREATE TABLE addresses (
    address_id BIGINT AUTO_INCREMENT PRIMARY KEY,
    customer_id BIGINT NOT NULL,
    address_type VARCHAR(20) NOT NULL DEFAULT 'SHIPPING',
    address_line VARCHAR(200) NOT NULL,
    city VARCHAR(80) NOT NULL,
    state_code VARCHAR(30) NOT NULL,
    postal_code VARCHAR(20) NOT NULL,
    country_code CHAR(2) NOT NULL DEFAULT 'IN',
    CONSTRAINT chk_address_type CHECK (address_type IN ('SHIPPING', 'BILLING')),
    CONSTRAINT fk_address_customer FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id) ON DELETE CASCADE
) ENGINE=InnoDB
"""
execute_sql(query)

### Create the `categories` entity

The category code is an alternate business key. The generated integer remains the compact primary key used by foreign keys.

In [ ]:
query = f"""
CREATE TABLE categories (
    category_id INT AUTO_INCREMENT PRIMARY KEY,
    category_code VARCHAR(30) NOT NULL UNIQUE,
    category_name VARCHAR(100) NOT NULL UNIQUE
) ENGINE=InnoDB
"""
execute_sql(query)

### Create the `products` entity

SKU is the product's business identifier. Price and active status are facts about one product, and the category foreign key implements a many-to-one relationship.

In [ ]:
query = f"""
CREATE TABLE products (
    product_id BIGINT AUTO_INCREMENT PRIMARY KEY,
    sku VARCHAR(40) NOT NULL UNIQUE,
    product_name VARCHAR(150) NOT NULL,
    category_id INT NOT NULL,
    current_price DECIMAL(12, 2) NOT NULL,
    is_active BOOLEAN NOT NULL DEFAULT TRUE,
    CONSTRAINT chk_product_price CHECK (current_price >= 0),
    CONSTRAINT fk_product_category FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
) ENGINE=InnoDB
"""
execute_sql(query)

### Create the `sales` entity

A sale has both a surrogate primary key and a unique order number used by people and external systems. The customer relationship is mandatory.

In [ ]:
query = f"""
CREATE TABLE sales (
    sale_id BIGINT AUTO_INCREMENT PRIMARY KEY,
    order_number VARCHAR(30) NOT NULL UNIQUE,
    customer_id BIGINT NOT NULL,
    ordered_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    sale_status VARCHAR(20) NOT NULL DEFAULT 'PLACED',
    CONSTRAINT chk_sale_status CHECK (sale_status IN ('PLACED', 'PAID', 'SHIPPED', 'CANCELLED')),
    CONSTRAINT fk_sale_customer FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id)
) ENGINE=InnoDB
"""
execute_sql(query)

### Create the `sale_items` associative entity

The composite primary key says a product appears at most once per sale. `unit_price` is stored here because the price charged is a fact about this sale line, not today's product price.

In [ ]:
query = f"""
CREATE TABLE sale_items (
    sale_id BIGINT NOT NULL,
    product_id BIGINT NOT NULL,
    quantity INT NOT NULL,
    unit_price DECIMAL(12, 2) NOT NULL,
    PRIMARY KEY (sale_id, product_id),
    CONSTRAINT chk_sale_item_quantity CHECK (quantity > 0),
    CONSTRAINT chk_sale_item_price CHECK (unit_price >= 0),
    CONSTRAINT fk_item_sale FOREIGN KEY (sale_id)
        REFERENCES sales(sale_id) ON DELETE CASCADE,
    CONSTRAINT fk_item_product FOREIGN KEY (product_id)
        REFERENCES products(product_id)
) ENGINE=InnoDB
"""
execute_sql(query)

### Create the `shipments` entity

A sale can be fulfilled by several shipments. Each shipment references the destination address actually selected for that fulfillment.

In [ ]:
query = f"""
CREATE TABLE shipments (
    shipment_id BIGINT AUTO_INCREMENT PRIMARY KEY,
    shipment_number VARCHAR(30) NOT NULL UNIQUE,
    sale_id BIGINT NOT NULL,
    ship_to_address_id BIGINT NOT NULL,
    carrier VARCHAR(60),
    tracking_number VARCHAR(100) UNIQUE,
    shipment_status VARCHAR(20) NOT NULL DEFAULT 'PENDING',
    shipped_at DATETIME NULL,
    CONSTRAINT chk_shipment_status CHECK (shipment_status IN ('PENDING', 'PACKED', 'SHIPPED', 'DELIVERED')),
    CONSTRAINT fk_shipment_sale FOREIGN KEY (sale_id) REFERENCES sales(sale_id),
    CONSTRAINT fk_shipment_address FOREIGN KEY (ship_to_address_id) REFERENCES addresses(address_id)
) ENGINE=InnoDB
"""
execute_sql(query)

## Load a small e-commerce dataset

These inserts use parameter placeholders for values. Placeholders prevent data from being interpreted as SQL and should be preferred over inserting user input into f-strings.

### Insert customers

Each tuple supplies one customer. `executemany` reuses the same parameterized statement efficiently.

In [ ]:
query = f"""
INSERT INTO customers (customer_number, customer_name, email)
VALUES (%s, %s, %s)
"""
execute_sql(query, [("CUST-1001", "Asha Retail", "asha@example.com"), ("CUST-1002", "Bilal Stores", "bilal@example.com")], many=True)

### Insert customer addresses

Customer 1 has separate shipping and billing addresses, demonstrating the one-to-many relationship.

In [ ]:
query = f"""
INSERT INTO addresses (customer_id, address_type, address_line, city, state_code, postal_code)
VALUES
    (1, 'SHIPPING', '10 Lake Road', 'Chennai', 'TN', '600001'),
    (1, 'BILLING', '22 Market Road', 'Chennai', 'TN', '600002'),
    (2, 'SHIPPING', '5 Tech Park', 'Hyderabad', 'TS', '500081')
"""
execute_sql(query)

### Insert product categories

Category codes are stable business-facing identifiers while numeric IDs support compact internal joins.

In [ ]:
query = f"""
INSERT INTO categories (category_code, category_name)
VALUES ('OFFICE', 'Office Supplies'), ('ELECTRONICS', 'Electronics')
"""
execute_sql(query)

### Insert products

Products refer to categories by foreign key. Their SKUs remain unique alternate keys.

In [ ]:
query = f"""
INSERT INTO products (sku, product_name, category_id, current_price)
VALUES
    ('NOTE-A5', 'A5 Notebook', 1, 80.00),
    ('PEN-BLU', 'Blue Pen Pack', 1, 120.00),
    ('LAMP-LED', 'LED Desk Lamp', 2, 1500.00)
"""
execute_sql(query)

### Insert sales

The order number is the business key customers see. The generated sale ID is used internally by related tables.

In [ ]:
query = f"""
INSERT INTO sales (order_number, customer_id, sale_status)
VALUES ('ORD-2026-0001', 1, 'PAID'), ('ORD-2026-0002', 2, 'PLACED')
"""
execute_sql(query)

### Insert sale items

The line rows resolve the sale-to-product many-to-many relationship and preserve the historical selling price.

In [ ]:
query = f"""
INSERT INTO sale_items (sale_id, product_id, quantity, unit_price)
VALUES (1, 1, 2, 80.00), (1, 2, 1, 120.00), (2, 3, 1, 1500.00)
"""
execute_sql(query)

### Insert a shipment

The shipment links one sale to the selected shipping address and stores logistics identifiers separately from the sale.

In [ ]:
query = f"""
INSERT INTO shipments (shipment_number, sale_id, ship_to_address_id, carrier, tracking_number, shipment_status, shipped_at)
VALUES ('SHIP-0001', 1, 1, 'Demo Express', 'TRK10001', 'SHIPPED', CURRENT_TIMESTAMP)
"""
execute_sql(query)

### Navigate the ER relationships

This set-based query walks foreign keys from customers through sales and lines to products, then calculates each line amount.

In [ ]:
query = f"""
SELECT
    s.order_number,
    c.customer_name,
    p.sku,
    p.product_name,
    si.quantity,
    si.unit_price,
    si.quantity * si.unit_price AS line_total
FROM sales AS s
JOIN customers AS c ON c.customer_id = s.customer_id
JOIN sale_items AS si ON si.sale_id = s.sale_id
JOIN products AS p ON p.product_id = si.product_id
ORDER BY s.order_number, p.sku
"""
execute_sql(query)

### Summarize one row per sale

Aggregation changes the grain from one row per sale item to one row per sale. Always know the intended grain of a query or table.

In [ ]:
query = f"""
SELECT
    s.order_number,
    c.customer_name,
    SUM(si.quantity * si.unit_price) AS sale_total
FROM sales AS s
JOIN customers AS c ON c.customer_id = s.customer_id
JOIN sale_items AS si ON si.sale_id = s.sale_id
GROUP BY s.sale_id, s.order_number, c.customer_name
ORDER BY s.order_number
"""
execute_sql(query)

## Normalization foundations

Normalization organizes relations using **functional dependencies**. If `A → B`, one value of A determines exactly one value of B. For example, `sku → product_name` in this business model.

Normalization reduces update, insert, and delete anomalies:

- **Update anomaly:** the same product name must be changed in many rows.
- **Insert anomaly:** a product cannot be recorded until somebody buys it.
- **Delete anomaly:** deleting the final sale accidentally removes the only product description.

The examples below are deliberately small. The final core model above is the design to keep.

## Unnormalized form

Unnormalized data may mix entities and contain repeating groups. The following anti-example stores multiple products as comma-separated text. MySQL cannot reliably enforce product foreign keys, quantities are difficult to aggregate, and individual items are awkward to update.

### Create an unnormalized sales table

This table is intentionally poor design. It exists only to make the normalization problems concrete.

In [ ]:
query = f"""
CREATE TABLE raw_sales_unnormalized (
    sale_number VARCHAR(30) PRIMARY KEY,
    customer_email VARCHAR(255) NOT NULL,
    customer_name VARCHAR(120) NOT NULL,
    products_csv VARCHAR(500) NOT NULL,
    quantities_csv VARCHAR(200) NOT NULL
) ENGINE=InnoDB
"""
execute_sql(query)

### Insert one repeating-group row

Two products and their quantities are packed into two text fields. Positional meaning now lives inside strings rather than relational constraints.

In [ ]:
query = f"""
INSERT INTO raw_sales_unnormalized
    (sale_number, customer_email, customer_name, products_csv, quantities_csv)
VALUES
    ('RAW-001', 'asha@example.com', 'Asha Retail', 'NOTE-A5,PEN-BLU', '2,1')
"""
execute_sql(query)

### Observe the unnormalized row

The row is readable to a person, but SQL cannot naturally join each embedded SKU to `products`.

In [ ]:
query = f"""
SELECT * FROM raw_sales_unnormalized
"""
execute_sql(query)

## First Normal Form — 1NF

A relation in **1NF** has atomic values for the chosen domain, no repeating column groups, and identifiable rows. Move each product occurrence to its own row.

The result below is easier to filter and aggregate, but customer and product facts repeat. Its composite key is `(sale_number, product_sku)`.

### Create the 1NF sales example

Every row represents one product within one sale. Values are atomic, but the table still combines sale, customer, and product facts.

In [ ]:
query = f"""
CREATE TABLE sales_1nf (
    sale_number VARCHAR(30) NOT NULL,
    sale_date DATE NOT NULL,
    customer_email VARCHAR(255) NOT NULL,
    customer_name VARCHAR(120) NOT NULL,
    product_sku VARCHAR(40) NOT NULL,
    product_name VARCHAR(150) NOT NULL,
    quantity INT NOT NULL,
    unit_price DECIMAL(12, 2) NOT NULL,
    PRIMARY KEY (sale_number, product_sku)
) ENGINE=InnoDB
"""
execute_sql(query)

### Insert atomic 1NF rows

The two products now occupy separate rows, enabling ordinary filtering and arithmetic.

In [ ]:
query = f"""
INSERT INTO sales_1nf
    (sale_number, sale_date, customer_email, customer_name, product_sku, product_name, quantity, unit_price)
VALUES
    ('SALE-001', '2026-08-03', 'asha@example.com', 'Asha Retail', 'NOTE-A5', 'A5 Notebook', 2, 80.00),
    ('SALE-001', '2026-08-03', 'asha@example.com', 'Asha Retail', 'PEN-BLU', 'Blue Pen Pack', 1, 120.00)
"""
execute_sql(query)

### Aggregate the 1NF rows

Atomic rows allow a normal `SUM`, even though duplicated descriptive facts still create maintenance risk.

In [ ]:
query = f"""
SELECT sale_number, SUM(quantity * unit_price) AS sale_total
FROM sales_1nf
GROUP BY sale_number
"""
execute_sql(query)

## Second Normal Form — 2NF

A table is in **2NF** when it is in 1NF and every non-key attribute depends on the **whole** candidate key—not only part of a composite key.

In `sales_1nf`, `sale_date` and customer information depend only on `sale_number`; product name depends only on `product_sku`. Those are partial dependencies on `(sale_number, product_sku)`. Split sale headers, products, and line items.

### Create 2NF products

Product attributes now have one home keyed by SKU, removing their partial dependency on the sale-line composite key.

In [ ]:
query = f"""
CREATE TABLE products_2nf (
    product_sku VARCHAR(40) PRIMARY KEY,
    product_name VARCHAR(150) NOT NULL
) ENGINE=InnoDB
"""
execute_sql(query)

### Create 2NF sale headers

Header facts depend on the sale number. Customer attributes remain here temporarily so the next normal form has a visible transitive dependency.

In [ ]:
query = f"""
CREATE TABLE sales_2nf (
    sale_number VARCHAR(30) PRIMARY KEY,
    sale_date DATE NOT NULL,
    customer_email VARCHAR(255) NOT NULL,
    customer_name VARCHAR(120) NOT NULL,
    customer_postal_code VARCHAR(20) NOT NULL,
    customer_city VARCHAR(80) NOT NULL
) ENGINE=InnoDB
"""
execute_sql(query)

### Create 2NF line items

Quantity and charged price depend on the complete `(sale_number, product_sku)` key. Foreign keys connect the separated facts.

In [ ]:
query = f"""
CREATE TABLE sale_items_2nf (
    sale_number VARCHAR(30) NOT NULL,
    product_sku VARCHAR(40) NOT NULL,
    quantity INT NOT NULL,
    unit_price DECIMAL(12, 2) NOT NULL,
    PRIMARY KEY (sale_number, product_sku),
    CONSTRAINT fk_2nf_item_sale FOREIGN KEY (sale_number) REFERENCES sales_2nf(sale_number),
    CONSTRAINT fk_2nf_item_product FOREIGN KEY (product_sku) REFERENCES products_2nf(product_sku)
) ENGINE=InnoDB
"""
execute_sql(query)

### Load 2NF products

Each product description is inserted once and can exist before its first sale.

In [ ]:
query = f"""
INSERT INTO products_2nf (product_sku, product_name)
VALUES ('NOTE-A5', 'A5 Notebook'), ('PEN-BLU', 'Blue Pen Pack')
"""
execute_sql(query)

### Load a 2NF sale header

The sale header is inserted once rather than repeated for each product line.

In [ ]:
query = f"""
INSERT INTO sales_2nf
    (sale_number, sale_date, customer_email, customer_name, customer_postal_code, customer_city)
VALUES
    ('SALE-001', '2026-08-03', 'asha@example.com', 'Asha Retail', '600001', 'Chennai')
"""
execute_sql(query)

### Load 2NF sale items

Only facts at the sale-line grain are stored in the associative table.

In [ ]:
query = f"""
INSERT INTO sale_items_2nf (sale_number, product_sku, quantity, unit_price)
VALUES ('SALE-001', 'NOTE-A5', 2, 80.00), ('SALE-001', 'PEN-BLU', 1, 120.00)
"""
execute_sql(query)

## Third Normal Form — 3NF

A table is in **3NF** when it is in 2NF and non-key attributes do not depend transitively on a key through another non-key attribute.

In the simplified 2NF header, `sale_number → customer_email → customer_name`, so the customer name does not belong in every sale header. If the business treats postal code as determining city, `customer_postal_code → customer_city` is another transitive dependency. Separate those facts.

Real postal codes can map to multiple localities, so this classroom dependency is a simplified business rule—not a universal geographic truth.

### Create the postal-code lookup

The example business rule gives each postal code one city. That determined fact now has one home.

In [ ]:
query = f"""
CREATE TABLE postal_codes_3nf (
    postal_code VARCHAR(20) PRIMARY KEY,
    city VARCHAR(80) NOT NULL
) ENGINE=InnoDB
"""
execute_sql(query)

### Create 3NF customers

Customer attributes depend on the customer key, while city is reached through the postal-code foreign key.

In [ ]:
query = f"""
CREATE TABLE customers_3nf (
    customer_id BIGINT AUTO_INCREMENT PRIMARY KEY,
    customer_email VARCHAR(255) NOT NULL UNIQUE,
    customer_name VARCHAR(120) NOT NULL,
    postal_code VARCHAR(20) NOT NULL,
    CONSTRAINT fk_3nf_customer_postal FOREIGN KEY (postal_code)
        REFERENCES postal_codes_3nf(postal_code)
) ENGINE=InnoDB
"""
execute_sql(query)

### Load the postal-code fact

The city is recorded once for the example postal code.

In [ ]:
query = f"""
INSERT INTO postal_codes_3nf (postal_code, city)
VALUES ('600001', 'Chennai')
"""
execute_sql(query)

### Load the 3NF customer

The customer stores only the postal-code reference, avoiding a repeated city value.

In [ ]:
query = f"""
INSERT INTO customers_3nf (customer_email, customer_name, postal_code)
VALUES ('asha@example.com', 'Asha Retail', '600001')
"""
execute_sql(query)

### Reconstruct normalized customer details

Normalization separates facts; joins reconstruct the business view when it is needed.

In [ ]:
query = f"""
SELECT c.customer_email, c.customer_name, p.postal_code, p.city
FROM customers_3nf AS c
JOIN postal_codes_3nf AS p ON p.postal_code = c.postal_code
"""
execute_sql(query)

## Boyce–Codd Normal Form — BCNF

BCNF is stronger than 3NF: for every non-trivial functional dependency `X → Y`, X must be a superkey.

Consider product-supplier terms where the business says each supplier always invoices in one currency. In a combined table, `supplier_code → currency_code`, but supplier code alone does not identify a product-supplier row. Separate supplier terms from the relationship.

### Create BCNF products

This small product table supplies one side of the product-supplier relationship.

In [ ]:
query = f"""
CREATE TABLE products_bcnf (
    product_sku VARCHAR(40) PRIMARY KEY,
    product_name VARCHAR(150) NOT NULL
) ENGINE=InnoDB
"""
execute_sql(query)

### Create BCNF supplier terms

Currency depends on the supplier key and is stored once per supplier.

In [ ]:
query = f"""
CREATE TABLE supplier_terms_bcnf (
    supplier_code VARCHAR(30) PRIMARY KEY,
    supplier_name VARCHAR(120) NOT NULL,
    currency_code CHAR(3) NOT NULL
) ENGINE=InnoDB
"""
execute_sql(query)

### Create the product-supplier relationship

Lead time depends on the complete product-supplier pair. Supplier currency is deliberately absent.

In [ ]:
query = f"""
CREATE TABLE product_supplier_terms_bcnf (
    product_sku VARCHAR(40) NOT NULL,
    supplier_code VARCHAR(30) NOT NULL,
    lead_time_days INT NOT NULL,
    PRIMARY KEY (product_sku, supplier_code),
    CONSTRAINT fk_bcnf_product FOREIGN KEY (product_sku) REFERENCES products_bcnf(product_sku),
    CONSTRAINT fk_bcnf_supplier FOREIGN KEY (supplier_code) REFERENCES supplier_terms_bcnf(supplier_code)
) ENGINE=InnoDB
"""
execute_sql(query)

## Fourth Normal Form — 4NF

**4NF** addresses independent multivalued dependencies. Suppose a customer can have several interests and several preferred contact channels, and every interest is independent of every channel.

Storing both lists in one table creates a cross-product: two interests × two channels produces four rows. Separate the independent multivalued facts.

### Create the 4NF anti-example

This table is valid relationally but combines two independent many-valued facts and therefore repeats combinations.

In [ ]:
query = f"""
CREATE TABLE customer_preferences_bad_4nf (
    customer_id BIGINT NOT NULL,
    interest_name VARCHAR(80) NOT NULL,
    channel_name VARCHAR(40) NOT NULL,
    PRIMARY KEY (customer_id, interest_name, channel_name)
) ENGINE=InnoDB
"""
execute_sql(query)

### Insert the multivalued cross-product

Two interests and two channels require four rows even though no interest is specifically related to a channel.

In [ ]:
query = f"""
INSERT INTO customer_preferences_bad_4nf (customer_id, interest_name, channel_name)
VALUES
    (1, 'Office', 'EMAIL'),
    (1, 'Office', 'SMS'),
    (1, 'Electronics', 'EMAIL'),
    (1, 'Electronics', 'SMS')
"""
execute_sql(query)

### Create independent customer interests

Each row now states exactly one customer-interest fact.

In [ ]:
query = f"""
CREATE TABLE customer_interests_4nf (
    customer_id BIGINT NOT NULL,
    interest_name VARCHAR(80) NOT NULL,
    PRIMARY KEY (customer_id, interest_name),
    CONSTRAINT fk_4nf_interest_customer FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
) ENGINE=InnoDB
"""
execute_sql(query)

### Create independent customer channels

Each row states one customer-channel fact without repeating interests.

In [ ]:
query = f"""
CREATE TABLE customer_channels_4nf (
    customer_id BIGINT NOT NULL,
    channel_name VARCHAR(40) NOT NULL,
    PRIMARY KEY (customer_id, channel_name),
    CONSTRAINT fk_4nf_channel_customer FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
) ENGINE=InnoDB
"""
execute_sql(query)

### Load normalized interests

Only two interest facts are required after the 4NF decomposition.

In [ ]:
query = f"""
INSERT INTO customer_interests_4nf (customer_id, interest_name)
VALUES (1, 'Office'), (1, 'Electronics')
"""
execute_sql(query)

### Load normalized channels

Only two independent channel facts are required.

In [ ]:
query = f"""
INSERT INTO customer_channels_4nf (customer_id, channel_name)
VALUES (1, 'EMAIL'), (1, 'SMS')
"""
execute_sql(query)

## Fifth Normal Form — 5NF

**5NF**, or project-join normal form, addresses join dependencies that require three or more projections. It is uncommon in ordinary application design.

Imagine approved fulfillment combinations of supplier, product, and warehouse. If business rules guarantee that a valid triple exists exactly when all three pairwise approvals exist, the ternary relation can be decomposed losslessly into three pair tables. If that guarantee is false, decomposing creates false combinations and the ternary table must remain.

### Create supplier-product approvals

This projection records which products each supplier is approved to provide.

In [ ]:
query = f"""
CREATE TABLE supplier_product_5nf (
    supplier_code VARCHAR(30) NOT NULL,
    product_sku VARCHAR(40) NOT NULL,
    PRIMARY KEY (supplier_code, product_sku)
) ENGINE=InnoDB
"""
execute_sql(query)

### Create supplier-warehouse approvals

This projection records which warehouses each supplier may serve.

In [ ]:
query = f"""
CREATE TABLE supplier_warehouse_5nf (
    supplier_code VARCHAR(30) NOT NULL,
    warehouse_code VARCHAR(30) NOT NULL,
    PRIMARY KEY (supplier_code, warehouse_code)
) ENGINE=InnoDB
"""
execute_sql(query)

### Create product-warehouse approvals

This projection records which products each warehouse can handle.

In [ ]:
query = f"""
CREATE TABLE product_warehouse_5nf (
    product_sku VARCHAR(40) NOT NULL,
    warehouse_code VARCHAR(30) NOT NULL,
    PRIMARY KEY (product_sku, warehouse_code)
) ENGINE=InnoDB
"""
execute_sql(query)

### Create a ternary comparison table

This table stores explicit triples for comparison. Whether it can be decomposed depends entirely on the stated business rule.

In [ ]:
query = f"""
CREATE TABLE supplier_product_warehouse_5nf (
    supplier_code VARCHAR(30) NOT NULL,
    product_sku VARCHAR(40) NOT NULL,
    warehouse_code VARCHAR(30) NOT NULL,
    PRIMARY KEY (supplier_code, product_sku, warehouse_code)
) ENGINE=InnoDB
"""
execute_sql(query)

### Load a supplier-product approval

The supplier is approved for the notebook product.

In [ ]:
query = f"""
INSERT INTO supplier_product_5nf (supplier_code, product_sku)
VALUES ('SUP-01', 'NOTE-A5')
"""
execute_sql(query)

### Load a supplier-warehouse approval

The supplier is approved to serve the Chennai warehouse.

In [ ]:
query = f"""
INSERT INTO supplier_warehouse_5nf (supplier_code, warehouse_code)
VALUES ('SUP-01', 'WH-CHN')
"""
execute_sql(query)

### Load a product-warehouse approval

The warehouse is approved to handle the notebook product.

In [ ]:
query = f"""
INSERT INTO product_warehouse_5nf (product_sku, warehouse_code)
VALUES ('NOTE-A5', 'WH-CHN')
"""
execute_sql(query)

### Reconstruct an approved fulfillment triple

Under the stated all-pairs business rule, joining the three projections reconstructs a valid supplier-product-warehouse combination.

In [ ]:
query = f"""
SELECT sp.supplier_code, sp.product_sku, sw.warehouse_code
FROM supplier_product_5nf AS sp
JOIN supplier_warehouse_5nf AS sw
    ON sw.supplier_code = sp.supplier_code
JOIN product_warehouse_5nf AS pw
    ON pw.product_sku = sp.product_sku
   AND pw.warehouse_code = sw.warehouse_code
"""
execute_sql(query)

## Denormalization and practical judgment

Normalization is a correctness tool, not a contest to create the most tables. Most transactional systems aim for **3NF or BCNF**, then measure actual workloads.

Deliberate denormalization may store derived totals, snapshots, or read-optimized copies when joins are demonstrably expensive. It adds synchronization responsibility: code, constraints, transactions, or pipelines must keep duplicated facts consistent.

Historical values are not necessarily denormalization mistakes. For example, `sale_items.unit_price` records the price agreed at purchase time; it is a fact about the sale line and should not change when `products.current_price` changes.

## Design review checklist

For every proposed table, ask:

- What business entity or relationship does one row represent?
- What is the row's grain?
- Which candidate keys exist, and which one should be primary?
- Which business keys still require `UNIQUE` constraints?
- Does every non-key attribute describe the key, the whole key, and nothing but the key?
- Are any independent lists packed into columns or multiplied together?
- Do foreign keys match the intended cardinality and optionality?
- Are history and current-state facts placed deliberately?
- Can important rules be enforced by constraints and transactions?

Good models evolve. Validate them with business examples, edge cases, query patterns, and realistic data volumes.

### Inspect the completed schema

The MySQL data dictionary provides a final inventory of the physical tables created by the model and normalization demonstrations.

In [ ]:
query = f"""
SELECT table_name, table_rows
FROM information_schema.tables
WHERE table_schema = %s AND table_type = 'BASE TABLE'
ORDER BY table_name
"""
execute_sql(query, (MYSQL_DATABASE,))

### Close the connection

Closing the connection releases its server-side resources. Rerun the connection cell before executing more SQL.

In [ ]:
if connection.is_connected():
    connection.close()
print("Connection closed.")